<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment2/audio_assignment_extract_birdnet_embeddings_speedup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q "birdnet[and-cuda]" scikit-learn pandas numpy tqdm joblib soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.3/123.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/3

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/audio_assignment")

AUDIO_DIR = PROJECT_DIR / "train_audio"

OUTPUT_DIR = PROJECT_DIR / "outputs_a1_birdnet_ml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Audio folder:", AUDIO_DIR)
print("Output folder:", OUTPUT_DIR)
print("Audio folder exists:", AUDIO_DIR.exists())



Audio folder: /content/drive/MyDrive/audio_assignment/train_audio
Output folder: /content/drive/MyDrive/audio_assignment/outputs_a1_birdnet_ml
Audio folder exists: True


In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

AUDIO_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".aiff", ".aif"]

allowed_exts = set([ext.lower() for ext in AUDIO_EXTENSIONS])

all_files_under_train_audio = [p for p in AUDIO_DIR.rglob("*") if p.is_file()]

accepted_files = []
ignored_files = []

for p in all_files_under_train_audio:
    ext = p.suffix.lower() if p.suffix else "[no extension]"

    if ext in allowed_exts:
        accepted_files.append(p)
    else:
        ignored_files.append(p)

print("Total files under train_audio:", len(all_files_under_train_audio))
print("Accepted audio files:", len(accepted_files))
print("Ignored files not in extension list:", len(ignored_files))

print("\nAllowed audio extensions:")
print(sorted(allowed_exts))

ignored_ext_counts = Counter(
    [p.suffix.lower() if p.suffix else "[no extension]" for p in ignored_files]
)

print("\nIgnored file extension counts:")
for ext, count in ignored_ext_counts.most_common():
    print(f"{ext}: {count}")

print("\nExample ignored files:")
for p in ignored_files[:30]:
    print(p)

ignored_df = pd.DataFrame({
    "filepath": [str(p) for p in ignored_files],
    "extension": [p.suffix.lower() if p.suffix else "[no extension]" for p in ignored_files]
})

ignored_csv_path = OUTPUT_DIR / "ignored_files_not_in_audio_extension_list.csv"
ignored_df.to_csv(ignored_csv_path, index=False)

print("\nIgnored file list saved to:")
print(ignored_csv_path)


# ----------------------------------------------------
# Create df for later train/validation split
# ----------------------------------------------------

rows = []

for audio_path in accepted_files:
    relative_path = audio_path.relative_to(AUDIO_DIR)

    # Expected:
    # train_audio/class_name/audio_file.wav
    if len(relative_path.parts) < 2:
        print("WARNING: File has no class folder, skipping:")
        print(audio_path)
        continue

    label = relative_path.parts[0]

    rows.append({
        "filepath": str(audio_path),
        "label": str(label)
    })

df = pd.DataFrame(rows)

print("\ndf created")
print("Total detected audio files in df:", len(df))
print("Total detected classes in df:", df["label"].nunique())

display(df.head())

df.to_csv(OUTPUT_DIR / "all_detected_audio_files.csv", index=False)

Total files under train_audio: 35549
Accepted audio files: 35549
Ignored files not in extension list: 0

Allowed audio extensions:
['.aif', '.aiff', '.flac', '.mp3', '.ogg', '.wav']

Ignored file extension counts:

Example ignored files:

Ignored file list saved to:
/content/drive/MyDrive/audio_assignment/outputs_a1_birdnet_ml/ignored_files_not_in_audio_extension_list.csv

df created
Total detected audio files in df: 35549
Total detected classes in df: 206


,filepath,label
0,/content/drive/MyDrive/audio_assignment/train_...,yecpar
1,/content/drive/MyDrive/audio_assignment/train_...,yecpar
2,/content/drive/MyDrive/audio_assignment/train_...,yecpar
3,/content/drive/MyDrive/audio_assignment/train_...,yecpar
4,/content/drive/MyDrive/audio_assignment/train_...,yecpar


In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import numpy as np

RANDOM_STATE = 42

# Store split outside individual experiment folder
# so A1, A2, B1, B2 all use the same validation set.
SPLIT_DIR = PROJECT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = SPLIT_DIR / "train_validation_split.csv"

# Keep this False normally.
# Set to True only if you intentionally want to delete/recreate the split.
RESET_SPLIT = False

def create_fixed_train_validation_split(df, split_file):
    """
    Creates a fixed train/validation split.

    Rules:
    - Classes with only 1 sample are forced into validation.
    - Classes with 2 samples get 1 train and 1 validation.
    - Classes with more than 2 samples use approximately 80/20 split.
    - The resulting split is saved to CSV and reused in future runs.
    """

    df = df.copy()
    df["filepath"] = df["filepath"].astype(str)
    df["label"] = df["label"].astype(str)

    counts = df["label"].value_counts()

    single_classes = counts[counts == 1].index.tolist()

    single_df = df[df["label"].isin(single_classes)].copy()
    multi_df = df[~df["label"].isin(single_classes)].copy()

    train_parts = []
    val_parts = []

    for label, group in multi_df.groupby("label"):
        group = group.sort_values("filepath").reset_index(drop=True)
        n = len(group)

        if n == 2:
            train_g, val_g = train_test_split(
                group,
                test_size=1,
                random_state=RANDOM_STATE,
                shuffle=True
            )
        else:
            train_g, val_g = train_test_split(
                group,
                test_size=0.2,
                random_state=RANDOM_STATE,
                shuffle=True
            )

        train_parts.append(train_g)
        val_parts.append(val_g)

    if len(train_parts) > 0:
        train_df = pd.concat(train_parts, ignore_index=True)
    else:
        train_df = pd.DataFrame(columns=df.columns)

    if len(val_parts) > 0:
        val_df = pd.concat(val_parts + [single_df], ignore_index=True)
    else:
        val_df = single_df.copy()

    train_df["split"] = "train"
    val_df["split"] = "validation"

    split_df = pd.concat([train_df, val_df], ignore_index=True)

    # Make output stable and readable
    split_df = split_df.sort_values(["split", "label", "filepath"]).reset_index(drop=True)

    split_df.to_csv(split_file, index=False)

    return split_df

if SPLIT_FILE.exists() and not RESET_SPLIT:
    print("Existing split found. Reusing fixed split:")
    print(SPLIT_FILE)

    split_df = pd.read_csv(SPLIT_FILE)

else:
    if RESET_SPLIT and SPLIT_FILE.exists():
        print("RESET_SPLIT=True, so recreating the split.")
    else:
        print("No existing split found. Creating split once.")

    split_df = create_fixed_train_validation_split(df, SPLIT_FILE)

print("Split loaded.")

display(split_df.head())
display(split_df["split"].value_counts())



train_df = split_df[split_df["split"] == "train"].copy().reset_index(drop=True)
val_df = split_df[split_df["split"] == "validation"].copy().reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Total samples:", len(split_df))

print("Train classes:", train_df["label"].nunique())
print("Validation classes:", val_df["label"].nunique())

single_example_classes = (
    split_df.groupby("label")
    .size()
    .loc[lambda x: x == 1]
    .index
    .tolist()
)

print("Single-example classes forced to validation:", len(single_example_classes))
print(single_example_classes[:20])

current_files = set(df["filepath"].astype(str))
split_files = set(split_df["filepath"].astype(str))

new_files_not_in_split = sorted(list(current_files - split_files))
missing_files_from_split = sorted(list(split_files - current_files))

print("New files not included in fixed split:", len(new_files_not_in_split))
print("Missing files from saved split:", len(missing_files_from_split))

if len(new_files_not_in_split) > 0:
    print("WARNING: These files are in the dataset folder but not in the saved split.")
    print("They will not be used unless you recreate the split intentionally.")
    print(new_files_not_in_split[:10])

if len(missing_files_from_split) > 0:
    print("WARNING: These files are in the saved split but missing from the dataset folder.")
    print("Check if files were moved, renamed, or deleted.")
    print(missing_files_from_split[:10])

Existing split found. Reusing fixed split:
/content/drive/MyDrive/audio_assignment/splits/train_validation_split.csv
Split loaded.


,filepath,label,split
0,/content/drive/MyDrive/audio_assignment/train_...,1161364,train
1,/content/drive/MyDrive/audio_assignment/train_...,1161364,train
2,/content/drive/MyDrive/audio_assignment/train_...,1161364,train
3,/content/drive/MyDrive/audio_assignment/train_...,1161364,train
4,/content/drive/MyDrive/audio_assignment/train_...,1161364,train


,count
split,
train,28357
validation,7192


Train samples: 28357
Validation samples: 7192
Total samples: 35549
Train classes: 202
Validation classes: 206
Single-example classes forced to validation: 4
['116570', '23150', '23724', '516975']
New files not included in fixed split: 0
Missing files from saved split: 0


In [ ]:
import birdnet

model = birdnet.load("acoustic", "2.4", "tf")

print("BirdNET model loaded")

BirdNET model loaded


In [ ]:
import numpy as np
import pandas as pd

def convert_birdnet_result_to_numpy(result):
    """
    Tries to convert BirdNET encode output into a numpy array.

    Expected final shape:
        windows x embedding_dimension
    """

    if isinstance(result, np.ndarray):
        arr = result

    elif isinstance(result, pd.DataFrame):
        numeric_df = result.select_dtypes(include=[np.number])
        arr = numeric_df.to_numpy()

    elif hasattr(result, "to_pandas"):
        pdf = result.to_pandas()
        numeric_df = pdf.select_dtypes(include=[np.number])
        arr = numeric_df.to_numpy()

    elif hasattr(result, "to_numpy"):
        arr = result.to_numpy()

    elif hasattr(result, "embeddings"):
        arr = np.asarray(result.embeddings)

    elif isinstance(result, dict):
        possible_keys = ["embeddings", "embedding", "features", "data"]
        arr = None

        for key in possible_keys:
            if key in result:
                arr = np.asarray(result[key])
                break

        if arr is None:
            raise ValueError(f"Could not find embeddings in dict keys: {result.keys()}")

    else:
        raise TypeError(f"Unknown BirdNET encode output type: {type(result)}")

    arr = np.asarray(arr)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    return arr


def extract_one_embedding(audio_path):
    """
    Extract BirdNET embeddings for one audio file.
    If the audio gives multiple window embeddings, average them.
    """

    result = model.encode(str(audio_path))
    arr = convert_birdnet_result_to_numpy(result)

    # Remove non-finite values if any
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    # Average window-level embeddings into clip-level embedding
    clip_embedding = arr.mean(axis=0)

    return clip_embedding

In [ ]:
test_file = train_df.iloc[0]["filepath"]

print("Testing file:", test_file)

test_embedding = extract_one_embedding(test_file)

print("Embedding shape:", test_embedding.shape)
print("First 10 values:", test_embedding[:10])

Testing file: /content/drive/MyDrive/audio_assignment/train_audio/1161364/iNat1216197.ogg
Embedding shape: (7, 1024)
First 10 values: [[0.1615618  0.33617735 0.5126887  ... 0.15418285 0.14871848 0.7309974 ]
 [0.3160696  0.18389042 0.48568058 ... 0.41137528 0.03535561 0.40468475]
 [0.21259405 0.2109555  0.45371893 ... 0.31539646 0.11101401 0.4475197 ]
 ...
 [0.1115047  0.43065727 0.6139331  ... 0.21575832 0.19744043 0.4791796 ]
 [0.13250501 0.3233242  0.5485172  ... 0.01217496 0.         0.17575872]
 [0.11028077 0.07299324 0.03307367 ... 0.9418027  0.17368317 0.9809286 ]]


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import hashlib
import json
from datetime import datetime

# Main folder where final embedding files will be saved
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings" / "birdnet_original"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Per-file cache folder. This makes extraction resumable.
CACHE_DIR = EMBEDDINGS_DIR / "per_file_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Embeddings will be saved to:")
print(EMBEDDINGS_DIR)

def make_cache_key(filepath):
    """
    Creates a stable filename-safe key for each audio file path.
    """
    filepath = str(filepath)
    return hashlib.sha1(filepath.encode("utf-8")).hexdigest()


def extract_embeddings_with_cache(dataframe, split_name):
    """
    Extracts BirdNET embeddings and saves:
    - one cache file per audio file
    - final X_<split>.npy
    - final y_<split>.npy
    - final meta_<split>.csv

    If cache files already exist, it loads them instead of extracting again.
    """

    split_cache_dir = CACHE_DIR / split_name
    split_cache_dir.mkdir(parents=True, exist_ok=True)

    X = []
    y = []
    paths = []
    cache_paths = []
    failed_rows = []

    for idx, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"Extracting {split_name} embeddings"
    ):
        audio_path = str(row["filepath"])
        label = str(row["label"])

        cache_key = make_cache_key(audio_path)
        cache_file = split_cache_dir / f"{cache_key}.npy"

        try:
            if cache_file.exists():
                emb = np.load(cache_file)
            else:
                emb = extract_one_embedding(audio_path)

                emb = np.asarray(emb, dtype=np.float32)

                if emb.ndim != 1:
                    emb = emb.reshape(-1)

                np.save(cache_file, emb)

            X.append(emb)
            y.append(label)
            paths.append(audio_path)
            cache_paths.append(str(cache_file))

        except Exception as e:
            print("Failed:", audio_path)
            print("Error:", e)

            failed_rows.append({
                "filepath": audio_path,
                "label": label,
                "error": str(e)
            })

    X = np.vstack(X).astype(np.float32)
    y = np.array(y)

    meta = pd.DataFrame({
        "filepath": paths,
        "label": y,
        "embedding_cache_file": cache_paths
    })

    # Final saved files
    X_path = EMBEDDINGS_DIR / f"X_{split_name}.npy"
    y_path = EMBEDDINGS_DIR / f"y_{split_name}.npy"
    meta_path = EMBEDDINGS_DIR / f"meta_{split_name}.csv"
    failed_path = EMBEDDINGS_DIR / f"failed_{split_name}.csv"

    np.save(X_path, X)
    np.save(y_path, y)
    meta.to_csv(meta_path, index=False)

    failed_df = pd.DataFrame(failed_rows)
    failed_df.to_csv(failed_path, index=False)

    print("\nSaved:")
    print(X_path)
    print(y_path)
    print(meta_path)
    print(failed_path)

    print("\nResult:")
    print(split_name, "X shape:", X.shape)
    print(split_name, "y shape:", y.shape)
    print(split_name, "failed files:", len(failed_rows))

    return X, y, meta, failed_df

X_train, y_train, meta_train, failed_train = extract_embeddings_with_cache(
    train_df,
    "train"
)

X_validation, y_validation, meta_validation, failed_validation = extract_embeddings_with_cache(
    val_df,
    "validation"
)

summary = {
    "created_at": datetime.now().isoformat(),
    "embedding_type": "BirdNET acoustic embeddings",
    "train_samples_saved": int(len(y_train)),
    "validation_samples_saved": int(len(y_validation)),
    "train_embedding_shape": list(X_train.shape),
    "validation_embedding_shape": list(X_validation.shape),
    "train_failed_files": int(len(failed_train)),
    "validation_failed_files": int(len(failed_validation)),
    "output_folder": str(EMBEDDINGS_DIR)
}

summary_path = EMBEDDINGS_DIR / "embedding_extraction_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Summary saved to:")
print(summary_path)

print(json.dumps(summary, indent=4))

Embeddings will be saved to:
/content/drive/MyDrive/audio_assignment/embeddings/birdnet_original


Extracting train embeddings:   0%|          | 0/28357 [00:00<?, ?it/s]